## Meta Learner
### Aligning the Predictions
Because the LSTM required a 30-day warmup, its test set is exactly 30 days shorter than the XGBoost test set. We must strictly align them so Day $t$ in XGBoost matches Day $t$ in the LSTM.

In [9]:
import numpy as np
import pandas as pd
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

print("Block 0: Rebuilding Models in Memory...")

# 1. Load Data
data = np.load("preprocessed_data.npz")

X_train_2d, X_val_2d, X_test_2d = data['X_train_2d'], data['X_val_2d'], data['X_test_2d']
y_train_cls, y_val_cls_3d, y_test_cls_3d = data['y_train_cls'], data['y_val_cls_3d'], data['y_test_cls_3d']

# 2. Re-Train XGBoost (Takes ~2 seconds)
ratio = float(np.sum(y_train_cls == 0)) / np.sum(y_train_cls == 1)
xgb_cls = xgb.XGBClassifier(
    objective='binary:logistic', n_estimators=1000, learning_rate=0.01, 
    max_depth=3, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=ratio, random_state=42
)
xgb_cls.fit(X_train_2d, y_train_cls, eval_set=[(X_val_2d, data['y_val_cls'])], verbose=False)
print("XGBoost Rebuilt.")

# 3. Re-Train LSTM (Takes ~15 seconds)
class AttentionLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim, dropout=0.3):
        super(AttentionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, 
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.attention_linear = nn.Linear(hidden_dim * 2, 1)
        self.fc1 = nn.Linear(hidden_dim * 2, 32)
        self.fc2 = nn.Linear(32, output_dim)
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention_linear(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.fc2(torch.relu(self.fc1(context)))

# Clean NaNs and prepare PyTorch Tensors
X_train_tensor = torch.nan_to_num(torch.tensor(data['X_train_3d'], dtype=torch.float32), nan=0.0)
y_train_tensor = torch.tensor(data['y_train_cls_3d'], dtype=torch.long)
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)

model = AttentionLSTM(input_dim=X_train_tensor.shape[2], hidden_dim=64, num_layers=2, output_dim=2)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

model.train()
# We use exactly 19 epochs since we proved earlier that this is the optimal stopping point
for epoch in range(19):
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

print("LSTM Rebuilt.")
print("Memory restored! You can now run Block 1, 2, and 3.")

Block 0: Rebuilding Models in Memory...
XGBoost Rebuilt.
LSTM Rebuilt.
Memory restored! You can now run Block 1, 2, and 3.


### Extracting Regime Features
We will extract the regime_200MA and vol_quartile_63d directly from the preprocessed arrays so the Meta-Learner knows what state the market is in.

In [11]:
import pandas as pd
import numpy as np
import torch

print("Combined Block 1 & 2: Generating Predictions & Extracting Regimes...")

LOOKBACK = 30 
data = np.load("preprocessed_data.npz")

# --- 1. Generate XGBoost Predictions ---
X_val_2d = data['X_val_2d']
X_test_2d = data['X_test_2d']
# Slice off the first 30 rows so XGBoost aligns perfectly with the LSTM
xgb_val_preds = xgb_cls.predict_proba(X_val_2d[LOOKBACK:])[:, 1]
xgb_test_preds = xgb_cls.predict_proba(X_test_2d[LOOKBACK:])[:, 1]

# --- 2. Generate LSTM Predictions ---
X_val_tensor = torch.nan_to_num(torch.tensor(data['X_val_3d'], dtype=torch.float32), nan=0.0)
X_test_tensor = torch.nan_to_num(torch.tensor(data['X_test_3d'], dtype=torch.float32), nan=0.0)

model.eval()
with torch.no_grad():
    lstm_val_preds = torch.softmax(model(X_val_tensor), dim=1)[:, 1].numpy()
    lstm_test_preds = torch.softmax(model(X_test_tensor), dim=1)[:, 1].numpy()

# --- 3. Extract Market Regimes ---
df = pd.read_csv("E:/fourth_sem/nifty_ml_hybrid/datasets/processed/nifty_engineered_features.csv")
df = df.dropna(subset=['target_ret_1d']).reset_index(drop=True)

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

val_df = df.iloc[train_end:val_end].iloc[LOOKBACK:].reset_index(drop=True)
test_df = df.iloc[val_end:].iloc[LOOKBACK:].reset_index(drop=True)

val_regime_200 = val_df['regime_200MA'].values
val_vol_quart = val_df['vol_quartile_63d'].values
test_regime_200 = test_df['regime_200MA'].values
test_vol_quart = test_df['vol_quartile_63d'].values

# --- 4. Stack into Final Meta-Learner Features ---
meta_X_val = np.column_stack((xgb_val_preds, lstm_val_preds, val_regime_200, val_vol_quart))
meta_X_test = np.column_stack((xgb_test_preds, lstm_test_preds, test_regime_200, test_vol_quart))

# Targets for the Meta-Learner
y_val = data['y_val_cls_3d']
y_test = data['y_test_cls_3d']

print(f"Meta-Feature Matrix Created! Shapes -> Val: {meta_X_val.shape}, Test: {meta_X_test.shape}")

Combined Block 1 & 2: Generating Predictions & Extracting Regimes...
Meta-Feature Matrix Created! Shapes -> Val: (373, 4), Test: (373, 4)


### 03.Training the Meta-Learner
We train a Logistic Regression model on the Validation set. It learns how much to trust XGBoost vs. LSTM under different volatility and trend regimes, and then we evaluate it on the untouched Test set.

In [13]:
# --- MISSING IMPORTS ADDED HERE ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("\nBlock 3: Training & Evaluating the Regime Meta-Learner...")

# Train the Meta-Learner on Validation Data
# We use class_weight='balanced' to ensure it respects both Up and Down days
meta_model = LogisticRegression(class_weight='balanced', random_state=42)
meta_model.fit(meta_X_val, y_val)

# Predict on strictly unseen Test Data
meta_test_preds = meta_model.predict(meta_X_test)

# Evaluate the final Hybrid Architecture
acc = accuracy_score(y_test, meta_test_preds)
print(f"\n--- HYBRID META-LEARNER EVALUATION ---")
print(f"Test Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, meta_test_preds, zero_division=0))

# Print Meta-Learner Coefficients (Crucial for Thesis Interpretability)
print("\nMeta-Learner Feature Weights:")
print(f"XGBoost Prediction Weight : {meta_model.coef_[0][0]:.4f}")
print(f"LSTM Prediction Weight    : {meta_model.coef_[0][1]:.4f}")
print(f"200-SMA Regime Weight     : {meta_model.coef_[0][2]:.4f}")
print(f"Volatility Regime Weight  : {meta_model.coef_[0][3]:.4f}")


Block 3: Training & Evaluating the Regime Meta-Learner...

--- HYBRID META-LEARNER EVALUATION ---
Test Accuracy: 54.69%

Classification Report:
              precision    recall  f1-score   support

           0       0.52      0.75      0.61       177
           1       0.62      0.37      0.46       196

    accuracy                           0.55       373
   macro avg       0.57      0.56      0.53       373
weighted avg       0.57      0.55      0.53       373


Meta-Learner Feature Weights:
XGBoost Prediction Weight : 2.7715
LSTM Prediction Weight    : -0.0312
200-SMA Regime Weight     : 0.6833
Volatility Regime Weight  : -0.1536


In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle
import os

print("Rebuilding and saving the Scaler...")

# Create your specific directory if it doesn't exist
save_dir = "E:/fourth_sem/nifty_ml_hybrid/saved_models"
os.makedirs(save_dir, exist_ok=True)

# 1. Load the original engineered features
df = pd.read_csv("E:/fourth_sem/nifty_ml_hybrid/datasets/processed/nifty_engineered_features.csv")

# 2. Identify the numerical columns (exactly as we did in preprocessing)
TARGET_COLS = ['target_ret_1d', 'target_dir_1d', 'target_quintile_1d']
CATEGORICAL_COLS = [
    'regime_200MA', 'vol_quartile_63d', 'is_monday', 'is_friday',
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos'
]
CATEGORICAL_COLS = [col for col in CATEGORICAL_COLS if col in df.columns]
NUMERICAL_COLS = [col for col in df.columns if col not in ['date'] + TARGET_COLS + CATEGORICAL_COLS]

# 3. Strictly isolate the Training Data (first 70%) to fit the scaler
n = len(df)
train_end = int(n * 0.70)
train_df = df.iloc[:train_end]

# 4. Fit the Scaler
scaler = StandardScaler()
scaler.fit(train_df[NUMERICAL_COLS])

# 5. Save it to your specific path
scaler_path = os.path.join(save_dir, "scaler.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)

print(f"✅ Scaler successfully rebuilt and saved to: {scaler_path}")

Rebuilding and saving the Scaler...
✅ Scaler successfully rebuilt and saved to: E:/fourth_sem/nifty_ml_hybrid/saved_models\scaler.pkl


### Saving all models

In [19]:
import os
import pickle
import torch

print("Exporting trained models to disk...")

# Create the models directory if it doesn't exist
os.makedirs("models", exist_ok=True)

# 1. Save XGBoost
# Location: .E:\fourth_sem\nifty_ml_hybrid\saved_models
xgb_cls.save_model("E:/fourth_sem/nifty_ml_hybrid/saved_models/xgb_model.json")
print("✅ Saved: models/xgb_model.json")

# 2. Save PyTorch LSTM Weights
# Location: ./models/lstm_weights.pth
torch.save(model.state_dict(), "E:/fourth_sem/nifty_ml_hybrid/saved_models/lstm_weights.pth")
print("✅ Saved: models/lstm_weights.pth")

# 3. Save Scikit-Learn Meta-Learner
# Location: ./models/meta_model.pkl
with open("E:/fourth_sem/nifty_ml_hybrid/saved_models/meta_model.pkl", "wb") as f:
    pickle.dump(meta_model, f)
print("✅ Saved: models/meta_model.pkl")

# 4. Save the Preprocessing Scaler (Crucial for live data)
# Location: ./models/scaler.pkl
# (Assuming 'scaler' is still in memory from preprocessing)
with open("E:/fourth_sem/nifty_ml_hybrid/saved_models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("✅ Saved: models/scaler.pkl")

Exporting trained models to disk...
✅ Saved: models/xgb_model.json
✅ Saved: models/lstm_weights.pth
✅ Saved: models/meta_model.pkl
✅ Saved: models/scaler.pkl


##